In [14]:
import polars as pl
import pandas as pd
import numpy as np
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import optuna
import lightgbm as lgb

C:\Users\HP\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [25]:
import json
import joblib

with open('../feature_selection/basic_magic_5_features.json' , 'r') as f:
    features = json.load(f)

lazy_df = pl.scan_parquet('../dataset/final_train_data/final_data.parquet')

last_cols = [c for c in lazy_df.columns if "_last" in c]
print(last_cols)

cols_to_load = features + ["target"]
df = lazy_df.select(cols_to_load).collect()

X = df.select(features).to_pandas()
y = df["target"].to_pandas()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)


def objective(trial):
    param = {
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 20, 100),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "lambda_l1": trial.suggest_float("lambda_l1", 1e-8, 10.0, log=True),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.4, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.4, 1.0),
        "bagging_freq": trial.suggest_int("bagging_freq", 1, 7),
    }

    model = LGBMClassifier(
        **param,
        n_estimators=2000, 
        objective="binary",
        metric="auc",
        is_unbalance=True,   
        device="cpu",          
        random_state=42,
        importance_type='gain',
        verbosity=-1
    )

    model.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        eval_metric='auc',
        callbacks=[
            lgb.early_stopping(stopping_rounds=100), 
        ]
    )

    y_pred = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_pred)
    
    return auc


study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler())
study.optimize(objective, n_trials=10) 

print("\n--- OPTIMIZATION COMPLETE ---")
print(f"Best AUC: {study.best_value:.4f}")
print("Best Params:", study.best_params)


C:\Users\HP\AppData\Local\Temp\ipykernel_2964\2331965812.py:9: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  last_cols = [c for c in lazy_df.columns if "_last" in c]


['P_2_last', 'D_39_last', 'B_1_last', 'B_2_last', 'R_1_last', 'S_3_last', 'D_41_last', 'B_3_last', 'D_42_last', 'D_43_last', 'D_44_last', 'B_4_last', 'D_45_last', 'B_5_last', 'R_2_last', 'D_46_last', 'D_47_last', 'D_48_last', 'D_49_last', 'B_6_last', 'B_7_last', 'B_8_last', 'D_50_last', 'D_51_last', 'B_9_last', 'R_3_last', 'D_52_last', 'P_3_last', 'B_10_last', 'D_53_last', 'S_5_last', 'B_11_last', 'S_6_last', 'D_54_last', 'R_4_last', 'S_7_last', 'B_12_last', 'S_8_last', 'D_55_last', 'D_56_last', 'B_13_last', 'R_5_last', 'D_58_last', 'S_9_last', 'B_14_last', 'D_59_last', 'D_60_last', 'D_61_last', 'B_15_last', 'S_11_last', 'D_62_last', 'D_63_last', 'D_64_last', 'D_65_last', 'B_16_last', 'B_17_last', 'B_18_last', 'B_19_last', 'D_66_last', 'B_20_last', 'D_68_last', 'S_12_last', 'R_6_last', 'S_13_last', 'B_21_last', 'D_69_last', 'B_22_last', 'D_70_last', 'D_71_last', 'D_72_last', 'S_15_last', 'B_23_last', 'D_73_last', 'P_4_last', 'D_74_last', 'D_75_last', 'D_76_last', 'B_24_last', 'R_7_last

[I 2026-03-07 00:31:18,043] A new study created in memory with name: no-name-8cb5b0e4-69f3-44da-8529-89769abb4f6a


Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[319]	valid_0's auc: 0.941537


[I 2026-03-07 00:31:24,119] Trial 0 finished with value: 0.9415368707585244 and parameters: {'learning_rate': 0.030393085894947353, 'num_leaves': 73, 'max_depth': 10, 'lambda_l1': 3.6769596349219396, 'feature_fraction': 0.5912816607513416, 'bagging_fraction': 0.6605037743358382, 'bagging_freq': 5}. Best is trial 0 with value: 0.9415368707585244.


...

Training until validation scores don't improve for 100 rounds


[I 2026-03-07 00:32:17,182] Trial 8 finished with value: 0.9415140429316637 and parameters: {'learning_rate': 0.04845883079369736, 'num_leaves': 34, 'max_depth': 9, 'lambda_l1': 3.433595150598651e-06, 'feature_fraction': 0.5091911191340193, 'bagging_fraction': 0.8967390480898224, 'bagging_freq': 2}. Best is trial 0 with value: 0.9415368707585244.


Early stopping, best iteration is:
[242]	valid_0's auc: 0.941514
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[837]	valid_0's auc: 0.941516


[I 2026-03-07 00:32:26,611] Trial 9 finished with value: 0.9415156290823934 and parameters: {'learning_rate': 0.01520278693804679, 'num_leaves': 37, 'max_depth': 5, 'lambda_l1': 0.007542499126264441, 'feature_fraction': 0.5938267089641291, 'bagging_fraction': 0.6057172100781376, 'bagging_freq': 6}. Best is trial 0 with value: 0.9415368707585244.



--- OPTIMIZATION COMPLETE ---
Best AUC: 0.9415
Best Params: {'learning_rate': 0.030393085894947353, 'num_leaves': 73, 'max_depth': 10, 'lambda_l1': 3.6769596349219396, 'feature_fraction': 0.5912816607513416, 'bagging_fraction': 0.6605037743358382, 'bagging_freq': 5}


In [27]:
best_model = LGBMClassifier(
        **study.best_params,
        n_estimators=2000, 
        objective="binary",
        metric="auc",
        is_unbalance=True,   
        device="cpu",          
        random_state=42,
        importance_type='gain',
        verbosity=-1
    )

best_model.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        eval_metric='auc',
        callbacks=[
            lgb.early_stopping(stopping_rounds=100), 
        ]
    )

y_pred = best_model.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_pred)
    
print(f"Final AUC: {auc}")

joblib.dump(best_model ,f"basic_model_{auc:.4f}.joblib")

Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[319]	valid_0's auc: 0.941537
Final AUC: 0.9415368707585244


['basic_model_0.9415.joblib']